In [ ]:
# Few-shot
# few: 소수, 몇개 안되는
# shot: 시도
# "give it a shot": 한번 해 봐~
# 문제를 잘 해결할 수 있게끔 유도를 하는 것

# Zero-shot: 프롬프트만으로 모델이 문제를 해결하는 방식
# One-shot: 한 개의 예시만 제공하고 문제를 해결하는 방식
# Few-shot: 2개 이상의 적은 수의 예시를 제공하고 문제를 해결하는 방식

# Prompt Engineering Research
# Too few (1-2)
# Optimal range (3-5)
# Too many (>5-10)

In [ ]:
# 주인공은 FewShotPromptTemplate


# 주요 속성
# prefix --- examples --- suffix

# Prefix (접두사): 문제의 범위나 수행해야 할 역할, 그리고 지켜야 할 규칙을 알려주는 역할
# Examples (예제): few examples (이런 입력에 이런 출력을 해주면 좋겠다)
# Suffix (접미사): 물어보고 싶은 실제 내용

# prefix는 안내문의 성격이므로 템플릿으로 변경을 할 필요가 없고
# suffix는 변수가 포함된 문자열 형태이며 값이 채워져서 LLM에 제출되는 최종 프롬프트
# 반면에 Examples는 템플릿 형태로 LLM에게 전달을 해야 함
# 이 모든 것들은 FewShotPromptTemplate으로 포장을 해야 함

# prefix, examples, suffix를 모두 하나로 묶어서 LLM으로 전달하는 것이 핵심

In [ ]:
import os

from dotenv import load_dotenv
load_dotenv()

import ssl
# SSL 검증을 비활성화하는 전역 설정
ssl._create_default_https_context = ssl._create_unverified_context
os.environ["CURL_CA_BUNDLE"] = ""
os.environ["SSL_CERT_FILE"] = ""
os.environ["PYTHONHTTPSVERIFY"] = "0"

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import FewShotPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
# FewShotPromptTemplate - 1

examples = [
    {"input": "이 영화 진짜 재미있었어요!", "output": "긍정"},
    {"input": "너무 맛이 없어서 남겼어요.", "output": "부정"},
    {"input": "평소에는 일찍 자는데 오늘은 늦게 잤네.", "output": "중립"},
]

# examples는 LLM이 이해할 수 없음
# PromptTemplate을 이용해서 LLM이 이해할 수 있는 형태로 변경

# example_prompt는 examples을 이용해서 어떻게 프롬프트를 만드는지를 기술하는 것
example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    # examples의 키를 변수로 설정해야 함 "input" → {input}, "output" → {output}
    template="사용자: {input}\n감정: {output}",
)

# FewShotPromptTemplate은 필요한 모든 것들을 하나로 포장
# 모델에게 실제로 수행하길 원하는 작업은 suffix
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="다음은 사용자가 말한 내용에 대한 감정 분류 예제입니다. 예제에 따라 입력된 문장의 감정을 긍정, 부정, 중립으로 분류하세요.",
    suffix="사용자: {input}\n감정:",
    # input_variables을 통해 검증
    # 템플릿에 {input}이 있는데 input_variables에 "input"이 있는지 없는지
    # input_variables에 "input"이 있는데 템플릿에 {input}이 있는지 없는지
    input_variables=["input"], 
)

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
chain = few_shot_prompt | llm

response = chain.invoke(input="오늘 날씨가 너무 좋아서 기분이 상쾌해.")
# input이라는 이름의 변수가 있어야 함
print("최종 응답:", response)

In [ ]:
# FewShotPromptTemplate - 2

# 1. 예제 정의 (학습 데이터)
examples = [
    {
        "input": "국내 증시가 반도체 수출 호조에 힘입어 상승 마감했다.",
        "output": "국내 증시, 반도체 수출, 상승 마감"
    },
    {
        "input": "새로운 인공지능 모델이 자연어 처리 성능을 크게 향상시켰다.",
        "output": "인공지능 모델, 자연어 처리, 성능 향상"
    },
    {
        "input": "유럽의 주요 도시들은 기후 변화에 대응하기 위해 친환경 정책을 적극적으로 추진하고 있다.",
        "output": "유럽, 기후 변화, 친환경 정책"
    }
]

# 2. 개별 예제 포맷팅 규칙 정의
example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="입력: {input}\n키워드: {output}"
)

# 3. 메인 Few-shot 프롬프트 포장
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="당신은 주어진 텍스트에서 핵심 키워드들을 추출하는 유용한 비서입니다. 중요한 단어나 구를 쉼표로 구분하여 나열하세요. 아래 예시를 참고하세요.\n\n",
    suffix="입력: {input}\n키워드:",
    input_variables=["input"]
)

# 4. LLM 모델 인스턴스 생성 및 체인 연결
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
chain = few_shot_prompt | llm

# 5. 체인 실행 및 응답 받기
response = chain.invoke({"input": "최근 스마트폰 시장은 폴더블폰의 등장으로 새로운 경쟁 구도를 맞이했다."})

print("최종 응답:", response.content)

In [ ]:
# partially format prompt templates

# 모든 변수가 채워지지 않은 일부만 완성된 상태의 템플릿
# 동적 프롬프트: 미리 알 수 있는 내용은 partial로 채워놓고 나머지는 실행 시간에 결정해서 채움

In [ ]:
from langchain_core.prompts import PromptTemplate

# 1. 모든 변수가 있는 원본 템플릿
full_template = PromptTemplate(
    input_variables=["language", "topic"],
    template="다음 {language} 주제에 대해 3가지로 설명해줘: {topic}"
)

# 2. 'language' 변수를 '한국어'로 미리 부분 할당
# invoke는 모든 변수의 값을 채우는 반면 partial은 일부 변수의 값만 채움
korean_template = full_template.partial(language="한국어")

# 3. 부분 포맷팅된 템플릿 사용
# 이제 'language' 변수 없이 'topic'만 전달
# response의 타입은 문자열로서 LLM에 전달될 최종 프롬프트가 됨
response = korean_template.format(topic="인공지능")

print(response)

In [ ]:
# partially format prompt templates + FewShotPromptTemplate

# 예제 정의 (영화 정보)
examples = [
    {
        "query": "영화 '인셉션'의 감독은?",
        "answer": "크리스토퍼 놀란입니다."
    },
    {
        "query": "영화 '기생충'의 개봉 연도는?",
        "answer": "2019년입니다."
    }
]

# 1. 
example_prompt = PromptTemplate(
    input_variables=["query", "answer"],
    template="질문: {query}\n답변: {answer}",
)

# 2. FewShotPromptTemplate
full_template = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="당신은 {role}입니다. 아래 예시를 참고하여 질문에 답해주세요.\n",
    suffix="질문: {query}\n답변:",
    input_variables=["role", "query"] # 'role'과 'query' 두 변수가 필요
)

# 3. partial() 함수를 사용해 'role'을 고정
expert_prompt = full_template.partial(role="영화 전문가")

# 4. LLM 모델 인스턴스 및 체인 생성
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
chain = expert_prompt | llm

print('format 확인:', expert_prompt.format(query="영화 'HOPE'의 개봉 연도는?"))
print('--------------------------')
# expert_prompt.format(query="영화 '어벤져스: 엔드게임'의 개봉 연도는?")을 실행하면
# expert_prompt = full_template.partial(role="영화 전문가")에 의해서
# full_template의 'role'은 값이 이미 할당이 되어 있음
# query="영화 '어벤져스: 엔드게임'의 개봉 연도는?"가 전달되면

# 전체 프롬프트는 prefix, examples, suffix 모두가 해당
# 당신은 영화 전문가입니다. 아래 예시를 참고하여 질문에 답해주세요.
# 질문: 영화 '인셉션'의 감독은?
# 답변: 크리스토퍼 놀란입니다.
# 질문: 영화 '기생충'의 개봉 연도는?
# 답변: 2019년입니다.
# 질문: 영화 '어벤져스: 엔드게임'의 개봉 연도는?

# 5. invoke() 호출하면서 'query'만 전달
response = chain.invoke({"query": "영화 '스타워즈'의 개봉 연도는?"})
print("최종 응답:", response.content)

In [ ]:
# PipelinePromptTemplate

# PromptTemplate이나 FewShotPromptTemplate은 하나의 입력을 만들어서 전달하는 것

# PipelinePromptTemplate은 여러 단계를 거치면서 입력이 가공이 됨
# 이전 단계의 프롬프트 출력이 다음 단계의 입력이 되는 구조
# 여러 개의 프롬프트 템플릿을 순차적으로 연결하여 복잡한 프롬프트를 만들 수 있음

In [ ]:
# PipelinePromptTemplate - 1    사용안함.
# from langchain_core.prompts import PromptTemplate
# from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_core.runnables import RunnableLambda

# # 1. 1단계: 페르소나를 정의하는 프롬프트
# full_prompt = PromptTemplate.from_template("당신은 전문 {role}입니다.")

# # 2. 2단계: 최종 질문에 해당하는 프롬프트
# final_prompt = PromptTemplate.from_template(
#     "{instruction}\n\n다음 질문에 답변해줘: {question}"
# )

# # 3. PipelinePromptTemplate 생성
# # pipeline_prompts: 중간 단계 템플릿들
# # final_prompt: 최종 프롬프트 템플릿
# # full_prompt > final_prompt의 순서로 진행


# # "instruction": "당신은 전문 요리사입니다."
# # "당신은 전문 요리사입니다.\n\n다음 질문에 답변해줘: 스테이크를 맛있게 굽는 법"

# from langchain_core.prompts import ChatPromptTemplate
# # 여러 프롬프트를 조합
# combined_template = ChatPromptTemplate.from_messages([
#     ("system", "You are a helpful assistant."),
#     ("human", "{formatted_input}")
# ])

# # from langchain_core.prompts.pipeline import PipelinePromptTemplate
# # 수정 후
# # from langchain_core.prompts import PipelinePromptTemplate
# from langchain_core.prompts import PipelinePromptTemplate
# pipeline_prompt = PipelinePromptTemplate(
#     final_prompt=final_prompt,
#     pipeline_prompts=[
#         ("instruction", full_prompt),
#         # ("key1", runnable1),
#         # ("key2", runnable2)
#         # ("key3", runnable3)
#         #full_prompt > runable1 > runable2 > runable3 > final_prompt 로 실행됨
#     ]
    
# )

# # 4. LLM 인스턴스 및 체인 생성
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
# chain = pipeline_prompt | llm

# response = chain.invoke(
#     {
#         "role": "요리사",
#         "question": "스테이크를 맛있게 굽는 법",
#     }
#     # 이 값은 pipeline_prompt의 full_prompt, final_prompt에 전달
#     # full_prompt의 결과는 'instruction'의 이름으로 접근이 가능하게 되고
#     # full_prompt > final_prompt에 의해서
#     # final_prompt에서 {instruction}의 값을 채울 수 있는 것
# )

# print(response.content)

In [ ]:
draft_prompt ("draft") >>> improve_prompt ("final_draft") >>> "{final_draft}"

In [ ]:
# PipelinePromptTemplate 대체 - Runnable 파이프라인 형태
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableLambda

# 1. 1단계: 페르소나를 정의하는 프롬프트
full_prompt = PromptTemplate.from_template("당신은 전문 {role}입니다.")

# 2. 2단계: 최종 질문에 해당하는 프롬프트
final_prompt = PromptTemplate.from_template(
    "{instruction}\n\n다음 질문에 답변해줘: {question}"
)

# 3. Runnable 파이프라인으로 변경
# 단계별 설명:
# 1) full_prompt를 실행하여 "instruction"에 결과를 저장
# 2) 원본 입력과 "instruction" 결과를 합쳐서 final_prompt에 전달
# 3) final_prompt 결과를 LLM에 전달

def step1_generate_instruction(inputs):
    """Step 1: 페르소나 프롬프트 실행"""
    # full_prompt 실행
    instruction = full_prompt.invoke({"role": inputs["role"]})
    # 원본 입력과 함께 반환 (step2에서 사용)
    return {
        "instruction": instruction,
        "question": inputs["question"],
    }

def step2_generate_final_prompt(inputs):
    """Step 2: 최종 프롬프트 생성"""
    # final_prompt 실행
    final_message = final_prompt.invoke({
        "instruction": inputs["instruction"],
        "question": inputs["question"]
    })
    return final_message

# Runnable 파이프라인 구성
pipeline_prompt = (
    RunnableLambda(step1_generate_instruction)
    | RunnableLambda(step2_generate_final_prompt)
)

# 4. LLM 인스턴스 및 체인 생성
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
chain = pipeline_prompt | llm

response = chain.invoke(
    {
        "role": "요리사",
        "question": "스테이크를 맛있게 굽는 법",
    }
)

print(response.content)


In [ ]:
# from langchain_core.prompts import PromptTemplate, PipelinePromptTemplate

# draft_template = """당신은 초보 작가입니다. 주어진 주제에 대한 짧은 초안을 작성해 주세요.
# 주제: {topic}
# """
# draft_prompt = PromptTemplate.from_template(draft_template)

# improve_template = """당신은 경험 많은 편집자입니다. 초안을 더 상세하고 풍부하게 개선해 주세요.
# 초안: {draft}
# """
# improve_prompt = PromptTemplate.from_template(improve_template)

# full_pipeline = PipelinePromptTemplate(
#     input_variables=["topic"],  # 파이프라인의 최종 입력 변수
#     pipeline_prompts=[
#         ("draft", draft_prompt),
#         ("final_draft", improve_prompt),
#     ],
#     final_prompt=PromptTemplate.from_template("{final_draft}"),
# )

# topic = "친환경 생활 습관"
# final_prompt = full_pipeline.format(topic=topic)

# print("--- 생성된 final_prompt ---")
# print(final_prompt)

In [ ]:
# PipelinePromptTemplate 대체 - Manual Chaining 형태
from langchain_core.prompts import PromptTemplate

# Step 1: 초안 작성 프롬프트
draft_template = """당신은 초보 작가입니다. 주어진 주제에 대한 짧은 초안을 작성해 주세요.
주제: {topic}
"""
draft_prompt = PromptTemplate.from_template(draft_template)

# Step 2: 초안 개선 프롬프트
improve_template = """당신은 경험 많은 편집자입니다. 초안을 더 상세하고 풍부하게 개선해 주세요.
초안: {draft}
"""
improve_prompt = PromptTemplate.from_template(improve_template)

# Manual Chaining - 단계별 실행
def process_writing_pipeline(topic):
    """작문 파이프라인 수동 처리"""
    
    # Step 1: 초안 생성
    draft = draft_prompt.invoke({"topic": topic}).to_string()
    print("--- Step 1: 초안 생성 ---")
    print(draft)
    print()
    
    # Step 2: 초안 개선
    final_draft = improve_prompt.invoke({"draft": draft}).to_string()
    print("--- Step 2: 초안 개선 ---")
    print(final_draft)
    print()
    
    return final_draft

# 실행
topic = "친환경 생활 습관"
final_prompt = process_writing_pipeline(topic)

print("--- 최종 결과 ---")
print(final_prompt)

In [ ]:
# PipelinePromptTemplate 대체 - Runnable 파이프라인 형태
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda

# Step 1: 초안 작성 프롬프트
draft_template = """당신은 초보 작가입니다. 주어진 주제에 대한 짧은 초안을 작성해 주세요.
주제: {topic}
"""
draft_prompt = PromptTemplate.from_template(draft_template)

# Step 2: 초안 개선 프롬프트
improve_template = """당신은 경험 많은 편집자입니다. 초안을 더 상세하고 풍부하게 개선해 주세��.
초안: {draft}
"""
improve_prompt = PromptTemplate.from_template(improve_template)

# Runnable 단계들 정의
def step1_generate_draft(inputs):
    """Step 1: 초안 생성"""
    draft = draft_prompt.invoke({"topic": inputs["topic"]})
    print("--- Step 1: 초안 생성 ---")
    print(draft.to_string())
    print()
    return {"draft": draft, "topic": inputs["topic"]}

def step2_improve_draft(inputs):
    """Step 2: 초안 개선"""
    final_draft = improve_prompt.invoke({"draft": inputs["draft"]})
    print("--- Step 2: 초안 개선 ---")
    print(final_draft.to_string())
    print()
    return final_draft

# Runnable 파이프라인 구성
writing_pipeline = (
    RunnableLambda(step1_generate_draft)
    | RunnableLambda(step2_improve_draft)
)

# 실행
topic = "친환경 생활 습관"
final_prompt = writing_pipeline.invoke({"topic": topic})

print("--- 최종 결과 ---")
print(final_prompt.to_string())

In [ ]:
# Runnable Mapping

# Runnable: 입력을 받아 처리하고 결과를 반환하는 가장 기본적인 실행 단위
# Runnable 인터페이스를 구현하며, 독립적으로 호출 가능

# 파이프(|) 연산자로 여러 Runnable을 연결하여 체인을 생성; prompt | llm
# 왼쪽 Runnable의 출력이 오른쪽 Runnable의 입력으로 전달되는 흐름
# Runnable의 출력은 단일 값에서부터 딕셔너리, 리스트와 같은 복잡한 형태도 포함

# RunnableMap은 여러 Runnable을 동시에 실행
# 그 결과는 하나의 딕셔너리로 묶어서 반환 → 여러 출력을 하나의 구조로 통합
# invoke로 전달된 입력값은 RunnableMap내의 Runnable에 동시에 전달


In [ ]:
# Runnable Mapping - 1

from langchain_core.runnables import RunnableMap
from langchain_core.prompts import ChatPromptTemplate


llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
prompt1 = ChatPromptTemplate.from_template("다음 주제에 대해 짧은 시를 써줘: {topic}")
prompt2 = ChatPromptTemplate.from_template("다음 주제에 대해 짧은 조언을 해줘: {topic}")

map_chain = RunnableMap(
    poem=prompt1 | llm,    # prompt1 | llm의 결과는 poem으로 접근
    advice=prompt2 | llm   # prompt2 | llm의 결과는 advice로 접근
)
# 아래와 동일 - 딕셔너리 형태
# map_chain = RunnableMap({
#     "poem": prompt1 | llm,
#     "advice": prompt2 | llm
# })

result = map_chain.invoke({"topic": "사랑"})
print(result['poem'].content)
print("=================================")
print(result['advice'].content)

In [ ]:

# Runnable Mapping - 2

from langchain_core.runnables import RunnablePassthrough, RunnableMap

# 아직 딕셔너리
runnable_dict = {
    "echo": RunnablePassthrough(),
    "uppercase": lambda x: x.upper()
}

# chain.invoke("hello")를 실행하면 "hello"가 RunnablePassthrough()와 lambda x: x.upper()에 전달
chain = RunnableMap(runnable_dict) | (lambda d: f"{d['echo']} → {d['uppercase']}")


print(chain.invoke("hello mr goh"))

In [ ]:
# Runnable Mapping - 3

from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnablePassthrough

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

topic_extraction_prompt = PromptTemplate.from_template(
    "다음 글의 핵심 주제를 한 단어로 요약해줘: {user_request}"
)
topic_extraction_chain = {"user_request": RunnablePassthrough()} | topic_extraction_prompt | llm
# {} | A | B
# {"user_request": RunnablePassthrough()}은 단순 딕셔너리
# {"user_request": RunnablePassthrough()} | 시점에서 {"user_request": RunnablePassthrough()}는 RunnableMap이 됨

topic_name = topic_extraction_chain.invoke(
    {"user_request": "인공지능의 윤리적인 문제점들에 대해 자세히 설명해줘."}
)
print(topic_name.content)

# 아래에서 계속

In [ ]:
# Runnable Mapping - 3

final_answer_prompt = PromptTemplate.from_template(
    "주제: {topic_name}\n\n위 주제에 대해 다음 요청에 답변해줘: {user_request}"
)

full_chain = {
    "topic_name": topic_extraction_chain,
    "user_request": RunnablePassthrough(),
} | final_answer_prompt | llm

final_response = full_chain.invoke(
    {"user_request": "인공지능의 윤리적인 문제점들에 대해 자세히 설명해줘."}
)

# 최종 프롬프트 확인
print("\n--- 최종 프롬프트 ---")

print(final_answer_prompt.format(
    topic_name=topic_name.content,
    user_request="인공지능의 윤리적인 문제점들에 대해 자세히 설명해줘."
))
print("\n--- 최종 응답 ---")
print(final_response.content)

In [ ]:
# Runnable Mapping - 4

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

prompt = ChatPromptTemplate.from_template(
    """다음 문맥(context)을 참고하여 질문(question)에 답하세요:

    <context>
    {context}
    </context>

    질문: {question}
    """
)

# RunnableMap을 사용하지 않고 딕셔너리로 체인을 구성합니다.
chain = {
    "context": RunnablePassthrough(),
    # {
    # "question": "피자의 유래에 대해 알려줘.",
    #"context": "피자는 이탈리아 남부 나폴리에서 유래된 음식입니다."
    # }
    "question": RunnablePassthrough()
    # {
    # "question": "피자의 유래에 대해 알려줘.",
    #"context": "피자는 이탈리아 남부 나폴리에서 유래된 음식입니다."
    # }
} | prompt | llm
# {} | A | B

print(chain.invoke({
    "question": "피자의 유래에 대해 알려줘.",
    "context": "피자는 이탈리아 남부 나폴리에서 유래된 음식입니다."
}))

In [ ]:
# Runnable Mapping - 4

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

prompt = ChatPromptTemplate.from_template(
    """다음 문맥(context)을 참고하여 질문(question)에 답하세요:

    <context>
    {context}
    </context>

    질문: {question}
    """
)

# RunnableMap을 사용하지 않고 딕셔너리로 체인을 구성합니다.
chain = {
    "context": RunnablePassthrough(), 
    "question": RunnablePassthrough()
} | prompt
# {} | A | B

print(chain.invoke({
    "question": "피자의 유래에 대해 알려줘.",
    "context": "피자는 이탈리아 남부 나폴리에서 유래된 음식입니다."
}))

In [ ]:
filled_prompt = prompt.invoke({
    "question": "피자의 유래에 대해 알려줘.",
    "context": "피자는 이탈리아 남부 나폴리에서 유래된 음식입니다."
})

print(filled_prompt)


In [ ]:
# RunnableParallel vs.	RunnableMap
# RunnableMap은 RunnableParallel의 alias
# 둘은 동일한 대상임

In [ ]:

# RunnableSequence는 여러 Runnable 객체를 순차적으로 연결하여 실행하는데 사용하는 클래스
# 파이프 연산자를 사용하여 여러 Runnable을 연결하면 자동으로 RunnableSequence 객체가 생성
# runnable1 | runnable2 | runnable3

In [ ]:
# Elements

# RunnableSequence
# Runnable 객체들을 순차적으로 연결
# 이전 단계의 출력이 다음 단계의 입력이 되는 직렬 파이프라인 만듬

# RunnableParallel(RunnableMap)
# 여러 개의 Runnable 객체들을 병렬로 실행하고, 그 결과를 딕셔너리 형태로 리턴

# RunnablePassthrough
# 입력값을 수정하지 않고 그대로 다음 Runnable로 전달하는 역할
# RunnablePassthrough.assign(a=b)을 사용하여 
# 입력 값을 그대로 + b의 결과에 이름표 a를 붙인 내용으로 딕셔너리를 만듬

# RunnableLambda
# 파이썬 함수를 Runnable 객체로 변환하여 체인에서 사용할 수 있게 함

# RunnableBranch
# 특정 조건에 따라 다른 Runnable 체인을 실행하는 데 사용

# RunnableWithFallbacks
# 기본 Runnable이 실패할 경우, 미리 정의된 대체(fallback) Runnable을 실행하도록 설정

In [ ]:
# RunnableBranch

In [ ]:
from langchain_core.runnables import RunnableBranch, RunnablePassthrough

# 두 개의 Runnable: name_chain, no_name_chain

# RunnablePassthrough에 전달되는 값은 x로
# {"name": "김민준"}
name_chain = RunnablePassthrough.assign(
    output=lambda x: f"안녕하세요, {x['name']}님!"
    # {"name": "김민준", 'output': "안녕하세요, 김민준님!}
)

# RunnablePassthrough에 전달되는 값은 x로
# {"city": "서울"}
no_name_chain = RunnablePassthrough.assign(
    output=lambda x: "안녕하세요, 익명의 사용자님!"
    # {"city": "서울", 'output': "안녕하세요, 익명의 사용자님!}
)

# RunnableBranch에 값 전달은 invoke()

# RunnableBranch의 기본 형태
# 조건 함수는 입력을 받아서 True/False를 리턴
# 조건 함수가 True가 되면 해당 Runnable을 실행
# 조건 함수가 False이면 다음 조건 함수를 평가
# 모든 조건 함수가 False가 되면 기본 Runnable을 실행

# RunnableBranch(
#     (조건 함수 1, 실행할 Runnable 1),
#     (조건 함수 2, 실행할 Runnable 2),
#     ...
#     기본값으로 실행할 Runnable


branch = RunnableBranch(
    # invoke로 전달한 값은 실행되는 runnable(name_chain, no_name_chain)에 전달됨
    # {"name": "김민준"}
    # {"city": "서울"}
    (lambda x: "name" in x, name_chain),
    no_name_chain
)

# 예제 실행 1: 'name' 키가 있을 때
result_with_name = branch.invoke({"name": "김민준"})
print(result_with_name)
# 결과: {'name': '김민준', 'output': '안녕하세요, 김민준님!'}

# 예제 실행 2: 'name' 키가 없을 때
result_without_name = branch.invoke({"city": "서울"})
print(result_without_name)
# 결과: {'city': '서울', 'output': '안녕하세요, 익명의 사용자님!'}

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# {"question": "서울 날씨 어때?"}
weather_chain = (
    PromptTemplate.from_template("오늘의 날씨를 알려줘. (도시: 서울)")
    | llm
)

time_chain = (
    PromptTemplate.from_template("현재 시간을 알려줘. (한국 표준시)")
    | llm
)

# 일반적인 질문
# {"question": "지금 몇 시야?"}
# {"question": "고양이와 강아지의 차이점은?"}
general_chain = (
    PromptTemplate.from_template("다음 질문에 친절하게 답변해줘: {question}")
    # "다음 질문에 친절하게 답변해줘: 지금 몇 시야?"
    # "다음 질문에 친절하게 답변해줘: 고양이와 강아지의 차이점은?"
    | llm
)

# RunnableBranch 정의
# 조건: 입력 텍스트에 "날씨" 또는 "시간" 키워드가 포함되어 있는지 확인
branch = RunnableBranch(
    # {"question": "지금 몇 시야?"}
    # {"question": "고양이와 강아지의 차이점은?"}
    (lambda x: "날씨" in x["question"], weather_chain),
    (lambda x: "시간" in x["question"], time_chain),
    general_chain # 모든 조건에 해당하지 않을 경우 실행될 기본값
)

weather_question = {"question": "서울 날씨 어때?"}
weather_result = branch.invoke(weather_question)
print("날씨 질문 결과:", weather_result)

time_question = {"question": "지금 몇 시야?"}
time_result = branch.invoke(time_question)
print("\n시간 질문 결과:", time_result)

# 일반적인 질문
general_question = {"question": "고양이와 강아지의 차이점은?"}
general_result = branch.invoke(general_question)
print("\n일반 질문 결과:", general_result)

In [ ]:
# 두 개의 질문을 두 개의 다른 LLM에게 보내서 답변을 병렬로 생성
# 두 개의 답변 중에서 더 짧은 답변을 선택하여 최종 결과로 선택

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda

# 1. 두 개의 다른 LLM 모델 정의
llm_A = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
llm_B = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# 2. RunnableParallel을 사용하여 두 모델을 병렬로 실행
# 입력: {"question": "..."}
# 출력: {"result_A": "...", "result_B": "..."}
parallel_chain = RunnableParallel(
    result_A=PromptTemplate.from_template("친절하게 답변해줘: {question}") | llm_A,
    result_B=PromptTemplate.from_template("친절하게 답변해줘: {question}") | llm_B
)

# 3. RunnableBranch를 사용하여 조건부 분기
# parallel_chain | branch_chain에서
# branch_chain가 넘겨 받은 입력은 {"result_A": "...", "result_B": "..."}의 형태
branch_chain = RunnableBranch(
    (
        lambda x: len(x['result_A'].content) < len(x['result_B'].content), 
        RunnableLambda(lambda x: x['result_A'])
    ),
    RunnableLambda(lambda x: x['result_B'])
)

# 4. RunnableSequence를 사용하여 전체 체인 구성
# 순서: 병렬 실행 -> 조건부 분기
full_chain = parallel_chain | branch_chain

# 예제 실행
question = {"question": "지구의 둘레는 어떻게 돼?"}
result = full_chain.invoke(question)

print(result.content)

In [ ]:
# 전처리 (사용자 키워드를 질문으로 변환)
# 병렬처리 (질문을 두가지 형태로 사용; 검색+위키피디아) → 검색+위키피디아는 실제 검색은 아님
# 후처리 (두 개의 결과를 하나로 병합)

from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnableLambda, RunnablePassthrough

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# 1. 전처리 단계: 키워드를 질문으로 바꿈

# RunnableLambda는 파이썬 함수를 Runnable로 변환
preprocess_chain = RunnableLambda(lambda x: {"question": f"{x['keyword']}에 대해 자세히 알려줘."})
# {"keyword": "아인슈타인"}가
# {'question': '아인슈타인에 대해 자세히 알려줘.'}로 바뀜

# 2. 병렬 처리 단계:

# RunnableParallel을 사용하여 두 체인을 동시에 실행

# ★★★★★★★★★
# 키 값은 두 단계(1); search_summary, wiki_summary
# parallel_chain을 실행하면 'search_summary'와 'wiki_summary'로 접근할 수 있는 결과 두 개가 나옴

# 키 값은 두 단계(2); search_result, wiki_result
# search_result=RunnableLambda, wiki_result=RunnableLambda에서 키 값 'search_result', 'wiki_result'

# 입력은 {'question': '아인슈타인에 대해 자세히 알려줘.'}
# 출력은 {"search_summary": "...", "wiki_summary": "..."}
parallel_chain = RunnableParallel(
    # 첫 번째 병렬 체인: 검색 결과 요약 (가상의 검색 도구)
    # RunnablePassthrough.assign는 결과에 키를 부여
    search_summary = RunnablePassthrough.assign(
        search_result=RunnableLambda(lambda x: f"가상의 검색 결과: {x['question']}에 대한 정보입니다.")
        # {'question': '아인슈타인에 대해 자세히 알려줘.'} +
        # {'search_result': "가상의 검색 결과: 아인슈타인에 대해 자세히 알려줘.에 대한 정보입니다."}
    ) | PromptTemplate.from_template("다음 검색 결과를 한 문장으로 요약해: {search_result}") | llm,
    # "다음 검색 결과를 한 문장으로 요약해: 가상의 검색 결과: 아인슈타인에 대해 자세히 알려줘.에 대한 정보입니다."
    
    # 두 번째 병렬 체인: 위키피디아 요약 (가상의 위키피디아 도구)
    # RunnablePassthrough.assign는 결과에 키를 부여
    wiki_summary = RunnablePassthrough.assign(
        wiki_result=RunnableLambda(lambda x: f"가상의 위키피디아 결과: {x['question']}에 대한 위키피디아 요약입니다.")
        # {'question': '아인슈타인에 대해 자세히 알려줘.'} +
        # {'wiki_result': '가상의 위키피디아 결과: 아인슈타인에 대해 자세히 알려줘.에 대한 위키피디아 요약입니다.'}
    ) | PromptTemplate.from_template("다음 위키피디아 요약을 한 문장으로 정리해: {wiki_result}") | llm
    # "다음 위키피디아 요약을 한 문장으로 정리해: 가상의 위키피디아 결과: 아인슈타인에 대해 자세히 알려줘.에 대한 위키피디아 요약입니다."
)

# 3. 후처리 단계: 병렬 실행 결과를 통합
# 입력: {"search_summary": "...", "wiki_summary": "..."}
# 출력: 최종 답변 문자열
postprocess_chain = RunnableLambda(
    lambda x: f"검색 요약: {x['search_summary'].content}\n위키피디아 요약: {x['wiki_summary'].content}"
)

# 4. 전체 체인 구성 (RunnableSequence)
# 순서: 전처리 -> 병렬 처리 -> 후처리
full_chain = preprocess_chain | parallel_chain | postprocess_chain
# preprocess_chain의 결과는 {"question": "아인슈타인에 대해 자세히 알려줘."}
# parallel_chain의 결과는 


# 예제 실행
keyword = {"keyword": "아인슈타인"}
result = full_chain.invoke(keyword)

print(result)